In [ ]:
#Import Statements Here
import os
import subprocess
import numpy as np
import pandas as pd
from statistics import mean
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as skl
from sklearn import metrics
from scipy import stats
rng = np.random.default_rng()
from sklearn import linear_model
from sklearn import metrics
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot
#from venn import venn

In [ ]:
#Need PD Version 2.2.3
print(pd.__version__)
#Need SKL Version 1.3.0
print(skl.__version__)
#Need NP Version 1.23.5
print(np.__version__)

In [ ]:
def getFromBucket(name_of_file_in_bucket):
    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file from the bucket to the current working space
    os.system(f"gsutil cp '{my_bucket}/data/{name_of_file_in_bucket}' .")

    print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
    # save dataframe in a csv file in the same workspace as the notebook
    my_dataframe = pd.read_csv(name_of_file_in_bucket)
    return my_dataframe.reset_index(drop=True)

In [ ]:
def storeInBucket(name_of_file_loc, a = 'dataFrame'):
    # get the bucket name
    
    # This snippet assumes you run setup first

    # This code saves your dataframe into a csv file in a "data" folder in Google Bucket

    # Replace df with THE NAME OF YOUR DATAFRAME
    if (type(name_of_file_loc)== type('string')):
        my_dataframe = pd.read_csv(name_of_file_loc)
        destination_filename = name_of_file_loc
    else:
        my_dataframe = name_of_file_loc
        destination_filename = a

    # Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
    

    ########################################################################
    ##
    ################# DON'T CHANGE FROM HERE ###############################
    ##
    ########################################################################

    # save dataframe in a csv file in the same workspace as the notebook
    my_dataframe.to_csv(destination_filename, index=False)

    # get the bucket name
    my_bucket = os.getenv('WORKSPACE_BUCKET')

    # copy csv file to the bucket
    args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
    output = subprocess.run(args, capture_output=True)

    # print output from gsutil
    output.stderr

In [ ]:
InnerMobileData = getFromBucket('InnerJoinMarch20MobileDevice.csv')
OuterMobileData = getFromBucket('OuterJoinMarch20MobileDevice.csv')
print(len(OuterMobileData))

In [ ]:
GeneticDataDf = getFromBucket("ACAFGeneticDataDf_2025Mar27.csv")
print(len(GeneticDataDf))

In [ ]:
DistributionDf = pd.DataFrame(columns = ['Genetic Variant','Percent Expression'])
for a in range(1,len(GeneticDataDf.columns)):
    Gene = GeneticDataDf.columns[a]
    Proportion = len(GeneticDataDf[GeneticDataDf.iloc[:,a]>0])/4803
    df2 = {'Genetic Variant':Gene,'Percent Expression':Proportion}
    #DistributionDf = DistributionDf.append(df2, ignore_index=True)

In [ ]:
CorrDf=GeneticDataDf.corr().drop(['chr16:89196249:G:GGCCGGCTGCCTGGCCGCCCTGCCGGCCGTGCAGGGTGTCCAGGA_GGCCGGCTGCCTGGCCGCCCTGCCGGCCGTGCAGGGTGTCCAGGA'])
CorrDf  = CorrDf.drop(['chr16:89196249:G:GGCCGGCTGCCTGGCCGCCCTGCCGGCCGTGCAGGGTGTCCAGGA_GGCCGGCTGCCTGGCCGCCCTGCCGGCCGTGCAGGGTGTCCAGGA'], axis =1)
CorrDf

Add in n3c cohort

In [ ]:
n3cCohort = getFromBucket('n3c_aou_cohort_ft_2024_Aug_08.csv').reset_index(drop=True)
print(len(n3cCohort))

Adding in EHR, survey data and Long Covid Status

In [ ]:
#Load in Long Covid EHR data from google bucket
LongCovidEHRDf = getFromBucket('ehr_model_result_feb_2025.csv')
#for a in LongCovidEHRDf.columns:
    #print(a)
print(len(LongCovidEHRDf[LongCovidEHRDf['long_covid']==1]))
print(len(LongCovidEHRDf))

In [ ]:
LongCovidEHRDf.drop('index', axis=1, inplace=True)

In [ ]:
#Load in Survey Data from google bucket
SurveyDataDF = getFromBucket('SurveyDataMarch2025.csv')
SurveyDataDF= SurveyDataDF.sort_values('person_id')
drop_list = ['Race: What Race Ethnicity', 'The Basics: Sexual Orientation',
             'Home Own: Current Home Own','Income: Annual Income','Education Level: Highest Grade','Living Situation: Stable House Concern']
SurveyDataDF = SurveyDataDF.drop(drop_list, axis=1)

In [ ]:
#Load in Genetic Data from google bucket
#currently this data is loaded in from the same notebook, so this cell is irrelevant until we do our sorting later on

GeneticDataDf = getFromBucket('ACAFGeneticDataDf_2025Mar27.csv')
GeneticDataDf['person_id'] = GeneticDataDf['IID']
GeneticDataDf= GeneticDataDf.sort_values('person_id')

Select Variables from Genetic data, Survey data, and EHR data to include in the model

In [ ]:
def selectVariables(df, variables):
    returnDf = df.loc[:, variables]
    returnDf=returnDf.sort_values('person_id', ignore_index=True)
    return returnDf

In [ ]:
#We select the data variables that we want from our EHR data added to our dataframe here

EHRAppendDf = selectVariables( LongCovidEHRDf, ['y_pred','person_id','long_covid'] )

In [ ]:
#We select the data variables that we want from our survey data added to our dataframe here

nonDuplicatesVariables = []

for a in SurveyDataDF.columns:
    if a in GeneticDataDf.columns:
        print(a)
    else: 
        nonDuplicatesVariables = nonDuplicatesVariables+ [a]
        
nonDuplicatesVariables = list(set(nonDuplicatesVariables+ ['person_id']) - set(['y_pred']))

In [ ]:
surveyAppendDf = selectVariables(SurveyDataDF, nonDuplicatesVariables)


Remove non-numeric Values

In [ ]:
def removeNonNumeric(df):
    #df = df[df['long_covid'] > - 1]
    df = df.replace('NaN', np.nan)
    df = df.replace('nan', np.nan)
    for a in df.columns:
    #columnDf = trialDf[a]
        columnMean = df.loc[df[a].notna() ,a].mean()
    #print(a)
    #print(columnMean)
        df[a] = df[a].fillna(value = columnMean)
    returnDf = df
    return returnDf

In [ ]:
geneAppendDf = selectVariables(GeneticDataDf, GeneticDataDf.columns)

Create a gold Standard

In [ ]:
#Here we are going to join just survey data and mobile device data

def joinMobileDeviceWithSurveyData(mobileDf, surveyDf, mergeType = 'inner'):
    
    returnDf = pd.merge(left = mobileDf, right = surveyDf, how = mergeType, left_on = 'person_id', right_on='person_id')
    
    return returnDf

In [ ]:
def joinMobileDeviceWithGeneticData(mobileDf, geneAppendDf, mergeType = 'inner'):
    
    returnDf = pd.merge(left = mobileDf, right = geneAppendDf, how = mergeType, left_on = 'person_id', right_on='person_id')
    
    return returnDf

In [ ]:
innerJoinSurveyMobileDevice = joinMobileDeviceWithSurveyData(InnerMobileData, surveyAppendDf, mergeType = 'outer')
innerJoinAllData = joinMobileDeviceWithGeneticData(innerJoinSurveyMobileDevice, geneAppendDf, mergeType = 'inner')
print(len(innerJoinAllData))

In [ ]:
goldStandardPersons = innerJoinAllData['person_id']

Combine data from all data sources by person_id

In [ ]:
#takes a bunch of dataframes, and joins them by the joining variable
    #duplicate columns will be dropped at the end, 
    #so if there are still duplicate column names, that means some of the columns don't match
def combineData(dataFrameList, joinVariable):
    joiningSet = dataFrameList[0].loc[:,joinVariable]
    
    
    #Now redefine joiningSet by going through each dataframe in the list 
    #and redefine as only the joiningVariables that are both in that dataframe, and the previous joiningSet
    for df in dataFrameList:
        joiningSet = df.loc[df[joinVariable].isin(joiningSet),joinVariable]
    
    
    #now that we have a joiningSet that includes only the common part of every data set, 
    #we need to restrict each data frame according to that set
    
    cleanedDfList = []
    for df in dataFrameList:
        cleanedDf = df.loc[df[joinVariable].isin(joiningSet),:].sort_values(joinVariable, ignore_index=True)
        cleanedDfList = cleanedDfList + [cleanedDf]
        
    #now join them all together with a concat statement and remove duplicate columns
    returnDf=pd.concat(cleanedDfList,axis=1)
    returnDf = returnDf.T.drop_duplicates().T
    return returnDf

In [ ]:
machineLearningDf = combineData([EHRAppendDf, surveyAppendDf, geneAppendDf], 'person_id')
machineLearningDf = machineLearningDf.reset_index(drop=True)

Adjacent: Combine data from Survey data and Mobile device data in a seperate dataframe

In [ ]:
#here we make a dataframe of just survey data and mobile device data with TONS of empty values
#We are going to apply this to a different algorithm that can handle these empty values
outerJoinSurveyMobileDevice = joinMobileDeviceWithSurveyData(OuterMobileData, surveyAppendDf, mergeType = 'outer')
#outerJoinSurveyMobileDevice= simplifyingAssumption(outerJoinSurveyMobileDevice)
#outerJoinSurveyMobileDevice = outerJoinSurveyMobileDevice.drop('min_covid_dt', axis=1)
outerJoinSurveyMobileDevice = outerJoinSurveyMobileDevice.reset_index(drop=True)

In [ ]:
Control model Data Frame with ALL data types

In [ ]:
#Append EHR, Genetic, Survey, and Mobile Device Data into a big dataframe with lots of wholes

#EHR Dataframe
LongCovidEHRDf = LongCovidEHRDf.sort_values('person_id').reset_index(drop=True)

#Survey Dataframe
SurveyDataDF = SurveyDataDF.sort_values('person_id').reset_index(drop=True)

#Genetic Dataframe
GeneticDataDf = GeneticDataDf.sort_values('person_id').reset_index(drop=True)

#Mobile Device Dataframe
OuterMobileData = OuterMobileData.sort_values('person_id').reset_index(drop=True)

#Use Existing Join function to join all the data
step1 = joinMobileDeviceWithSurveyData(LongCovidEHRDf, SurveyDataDF, mergeType = 'outer')
step2 = joinMobileDeviceWithSurveyData(step1, GeneticDataDf, mergeType = 'outer')
step3 = joinMobileDeviceWithSurveyData(step2, OuterMobileData, mergeType = 'outer')

ControlModelAllDataDf = step3.reset_index(drop=True)

In [ ]:
all_non_ehr = joinMobileDeviceWithSurveyData(outerJoinSurveyMobileDevice, GeneticDataDf, mergeType = 'outer')

Create Testing and Training data

In [ ]:
#We need to remove gold standard persons using the person list from the gold standard cohort
#If there's not enough genetic data to train on, we can split up the gold standard cohort and just reserve a part of it for later
#Alter the code below....

def makeTestAndTrainSets(df, trainProp):
    #First, control for the df being in the n3c cohort
    
    df = df[df["person_id"].isin(n3cCohort['person_id'])]
    
    trainDf = df[df['long_covid']==0].sample(frac=trainProp, random_state=101)
    trainDf =pd.concat([trainDf,df[df['long_covid']==1].sample(frac=trainProp, random_state=101)], ignore_index=True)
    
    NotInList=[]
    
    testDf =  df


    toDropList = df.index[df['person_id'].isin(trainDf['person_id'].tolist() + NotInList)]

    #print(toDropList)
    testDf = testDf.drop(index = toDropList, axis=0)
    testDf = testDf.reset_index(drop=True)
    trainDf = trainDf.reset_index(drop= True)
    return trainDf, testDf

In [ ]:
#Apply the function twice, once to make a gold standard cohort, and the other to make training and testing data\
#***Just do the split for testing and trining data based on all data available***
machineLearningDf, goldStandardDf = makeTestAndTrainSets(machineLearningDf, 0.7)

In [ ]:
#If we need to remove extra pieces from the training or test sets, we can use this list
trainDf, testDf = makeTestAndTrainSets(machineLearningDf, 0.5/0.7)

In [ ]:
#We need to make a test cohort of people with both survey and mobile device data
XGBtestSet=outerJoinSurveyMobileDevice
XGBtestSet =  XGBtestSet[XGBtestSet['person_id'].isin(goldStandardDf['person_id'])]

create training and test data for the control model (all data)

In [ ]:
#remove anywhere where longcovid is null or NA
ControlModelAllDataDf = ControlModelAllDataDf.dropna(subset = ['person_id', 'long_covid'])
ControlModelAllDataDf['sex_male'] = ControlModelAllDataDf['sex_male'].astype(int)

In [ ]:
#ControlModelAllDataDf
#first take out the gold standard cohort, then make training and testing data

GoldStandardControl = ControlModelAllDataDf[ControlModelAllDataDf['person_id'].isin(goldStandardDf['person_id'])]

all_list = list(ControlModelAllDataDf['person_id'])
remove_list = list(goldStandardDf['person_id'])
nonGoldControls = [i for i in all_list if i not in remove_list]

print(len(nonGoldControls))
print(len(ControlModelAllDataDf))

In [ ]:
nonGoldControlDf = ControlModelAllDataDf[ControlModelAllDataDf['person_id'].isin(nonGoldControls)]
trainControlDf, testControlDf = makeTestAndTrainSets(nonGoldControlDf, 0.5/0.7)

In [ ]:
GoldStandardControl = GoldStandardControl.reset_index(drop=True)

Save to bucket

Save the entire full joined data set

In [ ]:
storeInBucket(machineLearningDf, a = 'GeneticDataDfOct2024.csv')
storeInBucket(GoldStandardControl, a = 'GoldStandardAllDataOct8.csv')

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = ControlModelAllDataDf

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'long_covid_all_patient_data_2_25_2025.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

Save the testing and Training person ID sets

In [ ]:
all_persons = list(ControlModelAllDataDf['person_id'])
cohorts_df = pd.DataFrame({'person_id':all_persons})

training_set_persons = list(trainDf['person_id'])
cohorts_df['train']=0
cohorts_df.loc[cohorts_df['person_id'].isin(training_set_persons), 'train'] = 1

testing_set_persons = list(testDf['person_id'])
cohorts_df['test']=0
cohorts_df.loc[cohorts_df['person_id'].isin(testing_set_persons), 'test'] = 1

gold_standard_persons = list(GoldStandardControl['person_id'])
cohorts_df['gold_standard']=0
cohorts_df.loc[cohorts_df['person_id'].isin(gold_standard_persons), 'gold_standard'] = 1
cohorts_df

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = cohorts_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'long_covid_person_id_sets_2_25_2025.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

Save the features into a dataframe

In [ ]:
features = list(ControlModelAllDataDf.columns)
survey_features = ['Living Situation: People Under 18',
 'Marital Status: Current Marital Status',
 'The Basics: Birthplace',
 'Employment: Employment Status',
 'Living Situation: How Many Living Years',
 'Insurance: Health Insurance',
 'Active Duty: Active Duty Serve Status',
 'Biological Sex At Birth: Sex At Birth',
 'Living Situation: How Many People',
 'Electronic Smoking: Electric Smoke Participant',
 'Hookah Smoking: Hookah Smoke Participant',
 'Smoking: 100 Cigs Lifetime',
 'Smokeless Tobacco: Smokeless Tobacco Participant',
 'Recreational Drug Use: Which Drugs Used',
 'Alcohol: Alcohol Participant',
 'Alcohol: Drink Frequency Past Year',
 'Cigar Smoking: Cigar Smoke Participant',
 'Overall Health: Outside Travel 6 Month',
 'Overall Health: Social Satisfaction',
 'Overall Health: Organ Transplant',
 'Overall Health: Medical Form Confidence',
 'Overall Health: Health Material Assistance',
 'Overall Health: General Quality',
 'Overall Health: General Physical Health',
 'Overall Health: General Mental Health',
 'Overall Health: General Health',
 'Overall Health: Everyday Activities',
 'Overall Health: Difficult Understand Info',
 'Overall Health: Average Pain 7 Days',
 'Overall Health: Average Fatigue 7 Days',
 'Overall Health: General Social',
 'Overall Health: Emotional Problem 7 Days']
mobile_device_features =['before_covid_avg_heart_rate_x',
 'after_covid_avg_heart_rate_x',
 'before_covid_min_heart_rate',
 'before_covid_max_heart_rate',
 'after_covid_min_heart_rate',
 'after_covid_max_heart_rate',
 'before_covid_median_minutes_asleep',
 'after_covid_median_minutes_asleep',
 'before_covid_median_steps',
 'after_covid_median_steps']
genetic_features = ['chr1:155127096:C:G_G',
 'chr1:155127096:C:T_T',
 'chr1:155203736:G:A_A',
 'chr3:45793925:G:A_A',
 'chr3:45838989:T:C_C',
 'chr3:45859597:C:T_T',
 'chr3:101705614:T:C_C',
 'chr3:101705614:TG:T_T',
 'chr3:101780431:T:A_A',
 'chr3:101780431:T:C_C',
 'chr6:31153455:T:C_C',
 'chr6:31153649:G:A_A',
 'chr6:33076153:A:G_G',
 'chr6:41515652:G:C_C',
 'chr6:41522644:G:A_A',
 'chr6:41522644:G:C_C',
 'chr6:41534945:A:C_C',
 'chr9:133273813:C:T_C',
 'chr10:79946568:A:G_G',
 'chr10:79946568:A:T_T',
 'chr11:1219991:G:A_A',
 'chr11:1219991:G:C_C',
 'chr11:1219991:G:T_T',
 'chr11:34507219:C:A_A',
 'chr11:34507219:C:G_G',
 'chr11:34507219:C:T_T',
 'chr12:112914354:T:C_T',
 'chr12:112936943:C:G_G',
 'chr12:112936943:C:T_C',
 'chr12:132564254:T:A_A',
 'chr12:132564254:T:C_T',
 'chr12:132564254:T:G_G',
 'chr16:89196249:G:A_A',
 'chr16:89196249:G:T_T',
 'chr17:45707983:T:A_A',
 'chr17:45707983:T:C_C',
 'chr17:45707983:T:G_G',
 'chr17:49863303:C:T_T',
 'chr19:4719431:G:A_A',
 'chr19:10355447:C:T_T',
 'chr19:48867352:G:T_T',
 'chr19:50379362:T:C_C',
 'chr21:33252612:A:G_A',
 'chrX:15602217:T:C_C']
ehr_features = n3c_features = [
        "difficulty_breathing",
        "apprx_age",
        "fatigue",
        "op_post_visit_ratio",
        "dyspnea",
        "ip_post_visit_ratio",
        "albuterol",
        "hospitalized",
        "sex_male",
        "fluticasone",
        "palpitations",
        "mental_disorder",
        "uncomplicated_asthma",
        "chronic_pain",
        "malaise",
        "chronic_fatigue_syndrome",
        "formoterol",
        "tachycardia",
        "metabolic_disease",
        "chest_pain",
        "inflammation_of_specific_body_organs",
        "impaired_cognition",
        "diarrhea",
        "acetaminophen",
        "dyssomnia",
        "anxiety_disorder",
        "cough",
        "anxiety",
        "muscle_pain",
        "interstitial_lung_disease",
        "migraine",
        "degenerative_disorder",
        "viral_lower_respiratory_infection",
        "promethazine",
        "deficiency_of_micronutrients",
        "asthma",
        "disorder_characterized_by_pain",
        "apixaban",
        "lesion_of_lung",
        "inflammation_of_specific_body_systems",
        "breathing_related_sleep_disorder",
        "chronic_nervous_system_disorder",
        "iopamidol","loss_of_sense_of_smell",
        "amitriptyline",
        "sleep_disorder",
        "pain_of_truncal_structure",
        "neurosis",
        "headache",
        "tracheobronchial_disorder",
        "communication_disorder",
        "amnesia",
        "hypoxemia",
        "lower_respiratory_infection_caused_by_sars_cov_2",
        "bleeding",
        "amoxicillin",
        "disorder_due_to_infection",
        "chronic_sinusitis",
        "pain_in_lower_limb",
        "furosemide",
        "buspirone",
        "vascular_disorder",
        "memory_impairment",
        "insomnia",
        "budesonide",
        "prednisone",
        "pneumonia_caused_by_sars_cov_2",
        "clavulanate",
        "dizziness",
        "neuropathy",
        "iron_deficiency_anemia_due_to_blood_loss",
        "estradiol",
        "ceftriaxone",
        "shoulder_joint_pain",
        "sexually_active",
        "abdominal_pain",
        "skin_sensation_disturbance",
        "ketorolac",
        "depressive_disorder",
        "hyperlipidemia",
        "chronic_kidney_disease_due_to_hypertension",
        "spondylosis",
        "vascular_headache",
        "fibrosis_of_lung",
        "acute_respiratory_disease",
        "chronic_cough",
        "osteoporosis",
        "lorazepam",
        "connective_tissue_disorder_by_body_site",
        "adjustment_disorder",
        "benzonatate",
        "shoulder_pain",
        "mineral_deficiency",
        "obesity",
        "epinephrine",
        "dependence_on_enabling_machine_or_device",
        "dependence_on_respiratory_device",
        "inflammation_of_specific_body_structures_or_tissue",
        "spironolactone",
        "cholecalciferol",
        "heart_disease",
        "pain",
        "major_depression__single_episode",
        "meloxicam",
        "hydrocortisone",
        "collagen_disease",
        "headache_disorder",
        "hypoxemic_respiratory_failure",
        "morphine",
        "cardiac_arrhythmia",
        "seborrheic_keratosis",
        "gabapentin",
        "dulaglutide",
        "hypertensive_disorder",
        "effusion_of_joint",
        "moderate_persistent_asthma",
        "morbid_obesity",
        "seborrheic_dermatitis",
        "rbc_count_low",
        "blood_chemistry_abnormal",
        "acute_digestive_system_disorder",
        "sars_cov_2__covid_19__vaccine__mrna_spike_protein",
        "influenza_b_virus_antigen",
        "pulmonary_function_studies_abnormal",
        "sleep_apnea",
        "abnormal_presence_of_protein",
        "sodium_chloride",
        "atropine",
        "aspirin",
        "cognitive_communication_disorder",
        "metronidazole",
        "ethinyl_estradiol",
        "gadopentetate_dimeglumine",
        "traumatic_and_or_non_traumatic_injury_of_anatomical_site",
        "colchicine",
        "anomaly_of_eye",
        "oxycodone",
        "osteoarthritis",
        "complication_of_pregnancy__childbirth_and_or_the_puerperium",
        "allergic_rhinitis",
        "dizziness_and_giddiness",
        "genitourinary_tract_hemorrhage",
        "duloxetine",
        "bipolar_disorder",
        "vitamin_disease",
        "respiratory_obstruction",
        "genuine_stress_incontinence",
        "chronic_disease_of_respiratory_system",
        "traumatic_and_or_non_traumatic_injury",
        "drug_related_disorder",
        "nortriptyline",
        "involuntary_movement",
        "knee_pain",
        "peripheral_nerve_disease",
        "gastroesophageal_reflux_disease_without_esophagitis",
        "mupirocin",
        "fluconazole",
        "pure_hypercholesterolemia",
        "kidney_disease",
        "injury_of_free_lower_limb",
        "glaucoma",
        "backache",
        "tachyarrhythmia",
        "myocarditis",
        "nitrofurantoin",
        "prediabetes",
        "sodium_acetate",
        "apnea",
        "losartan",
        "radiology_result_abnormal",
        "pantoprazole",
        "hemoglobin_low",
        "mixed_hyperlipidemia",
        "mass_of_soft_tissue",
        "levonorgestrel",
        "omeprazole",
        "allergic_disposition",
        "metformin",
        "fentanyl",
        "spinal_stenosis_of_lumbar_region",
        "cyst",
        "soft_tissue_lesion",
        "altered_bowel_function",
        "skin_lesion",
        "triamcinolone",
        "pain_in_upper_limb",
        "acute_respiratory_infections",
        "neck_pain",
        "guaifenesin",
        "disorders_of_initiating_and_maintaining_sleep",
        "loratadine",
        "vitamin_b12",
        "hypercholesterolemia",
        "potassium_chloride",
        "arthropathy",
        "chronic_kidney_disease_due_to_type_2_diabetes_mellitus",
        "disease_of_non_coronary_systemic_artery",
        "soft_tissue_injury",
        "cytopenia"
]

In [ ]:
features_df = pd.DataFrame({'feature_name':features})

features_df['ehr_feature'] = 0
features_df.loc[features_df['feature_name'].isin(ehr_features), 'ehr_feature'] = 1

features_df['survey_feature'] = 0
features_df.loc[features_df['feature_name'].isin(survey_features), 'survey_feature'] = 1

features_df['mobile_device_feature'] = 0
features_df.loc[features_df['feature_name'].isin(mobile_device_features), 'mobile_device_feature'] = 1

features_df['genetic_feature'] = 0
features_df.loc[features_df['feature_name'].isin(genetic_features), 'genetic_feature'] = 1

features_df

In [ ]:
# This snippet assumes you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe = features_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename = 'long_covid_features_2_25_2025.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# save dataframe in a csv file in the same workspace as the notebook
my_dataframe.to_csv(destination_filename, index=False)

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file to the bucket
args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/data/"]
output = subprocess.run(args, capture_output=True)

# print output from gsutil
output.stderr

save all data dataframe

In [ ]:
#ControlModelAllDataDf
all_data_patients = ControlModelAllDataDf[ControlModelAllDataDf['person_id'].isin(goldStandardPersons)]
len(all_data_patients)

In [ ]:
all_data_patients = all_data_patients.loc[:,survey_features + ehr_features+ genetic_features+ ['person_id','long_covid'] ]

In [ ]:
storeInBucket(all_data_patients, a = 'patients_with_all_data_march_31_2025.csv')

Make Demographic table for these patients

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import plotnine
from plotnine import *  # Provides a ggplot-like interface to matplotlib.

# Get the BigQuery curated dataset for the current workspace context.
CDR = os.environ['WORKSPACE_CDR']

## Plot setup.
theme_set(theme_bw(base_size = 11)) # Default theme for plots.

def get_boxplot_fun_data(df):
  """Returns a data frame with a y position and a label, for use annotating ggplot boxplots.

  Args:
    d: A data frame.
  Returns:
    A data frame with column y as max and column label as length.
  """
  d = {'y': max(df), 'label': f'N = {len(df)}'}
  return(pd.DataFrame(data=d, index=[0]))

## ---------------[ CHANGE THESE AS NEEDED] ---------------------------------------
# Set default parameter values so that all snippets run successfully with no edits needed.
COHORT_QUERY = f'SELECT person_id FROM `{CDR}.person`'  # Default to all participants.
MEASUREMENT_OF_INTEREST = 'hemoglobin'
# Tip: the next four parameters could be set programmatically using one row from
# the result of measurements_of_interest_summary.sql
MEASUREMENT_CONCEPT_ID = 3004410        # Hemoglobin A1c
UNIT_CONCEPT_ID = 8554                  # percent
MEASUREMENT_NAME = '<this should be the measurement name>'
UNIT_NAME = '<this should be the unit name>'

In [ ]:
total_number_of_participants_df = pd.io.gbq.read_gbq(f'''

-- Compute the count of unique participants in our All of Us cohort.
SELECT
  person_id, 
  self_reported_category_concept_id,
  self_reported_category_source_concept_id,
  self_reported_category_source_value, year_of_birth, gender_source_value, race_source_value, ethnicity_source_value, sex_at_birth_source_value
  --*
FROM
  `{CDR}.person`
WHERE
  person_id IN ({COHORT_QUERY})

''',
  dialect='standard')

total_number_of_participants_df['age']= 2025 - total_number_of_participants_df['year_of_birth']

In [ ]:
demographic_data_df = pd.merge(all_data_patients, total_number_of_participants_df, on='person_id', how='inner')

In [ ]:
case_demo_df = demographic_data_df[demographic_data_df['long_covid']==1]
control_demo_df = demographic_data_df[demographic_data_df['long_covid']==0]

In [ ]:
demographic_data_df['self_reported_category_source_value'].describe()

In [ ]:
control_demo_df['self_reported_category_source_value'].value_counts()

In [ ]:
case_demo_df['self_reported_category_source_value'].value_counts()